# VGG-style CNN references — RGB and grayscale

Trains and **saves** the two CNN reference rows for the QRC classification study
(`notes/qrc_classification_setup.md`, item 4). They are context for a land-cover
audience, deliberately **outside** the ablation: a CNN sees the full 64×64 image
with no PCA bottleneck, so it measures what the d = 18 cut costs, not whether the
reservoir helps.

Two variants, same architecture, same split, same schedule:

- **RGB** — where the literature lives, and what `Classificaton_EuroSAT_VGG.ipynb`
  already did (that notebook is RGB despite the QRC pipeline being grayscale).
- **Grayscale** — the one that speaks to the bottleneck. Without it, the gap
  between 18 PCs and the CNN mixes two effects, colour and spatial resolution,
  and the discussion cannot separate them.

**The split is the QRC split.** Same ordering, same seeds, same stratification as
`QRC_Embeddings_Clean.ipynb`, so the models are evaluated on exactly the 2700 test
rows the QRC arms are scored on — which is what makes a paired test between a CNN
row and a QRC row legitimate later.

Note the older notebook used a 2-way split (21600 / 5400) with no validation
split, trained a flat 5 epochs and never saved the model — its reported 0.8794 is
on a 5400-image superset of this test set and cannot be reproduced. This notebook
replaces it as the source of the reference numbers.

**Outputs** under `models/vgg_classifier/`:

| file | what |
|---|---|
| `vgg_{rgb,gray}.keras` | trained model |
| `vgg_{rgb,gray}_history.json` | training curves |
| `vgg_test_predictions.npz` | per-image predicted class and probabilities on the QRC test rows, both variants |

The `.npz` is the one that matters downstream: paired tests (McNemar) need
per-image outcomes, not summary accuracies.

In [1]:
import json
import os
import random
import subprocess
import sys
from pathlib import Path

os.environ.setdefault("TF_CPP_MIN_LOG_LEVEL", "3")

import numpy as np
import tensorflow as tf
from PIL import Image
from sklearn.model_selection import train_test_split

sys.path.insert(0, str(Path.cwd() / "train_scripts"))
import run_eurosat_qrc_pipeline as qrc      # dataset paths, to_grayscale, SEED

SEED = qrc.SEED                             # 42 — same split as the QRC arms
OUT_DIR = Path("models/vgg_classifier")     # note: the old notebook used the
OUT_DIR.mkdir(parents=True, exist_ok=True)  # misspelled models/vgg_classfier
LABELS_PATH = qrc.MODEL_DIR / f"split_labels_seed{SEED}.npz"

EPOCHS, PATIENCE, BATCH = 50, 10, 32
# Patience 10, not 5: with rmsprop and no augmentation the validation loss swings
# by more than the gap between the two variants (rgb went
# 0.676 0.504 0.621 0.446 0.458 ...), and patience 5 stopped the RGB run at epoch
# 4 while grayscale reached a four-epoch plateau. That made the rgb-vs-gray
# comparison an artefact of when each run happened to be cut off.
RETRAIN = True                              # flip back to False once trained:
                                            # True overwrites the saved models

random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)
tf.config.experimental.enable_tensor_float_32_execution(False)
for _g in tf.config.list_physical_devices("GPU"):
    tf.config.experimental.set_memory_growth(_g, True)

# This card is 4 GB and the reservoir embedding run takes ~2 GB of it. Training a
# CNN alongside it will OOM -- check before starting, not three epochs in.
try:
    _free = int(subprocess.run(
        ["nvidia-smi", "--query-gpu=memory.free", "--format=csv,noheader,nounits"],
        capture_output=True, text=True, check=True).stdout.split("\n")[0])
    print(f"GPU free: {_free} MiB")
    if _free < 1800:
        print("  WARNING: little free memory. Is QRC_Embeddings_Clean still running?\n"
              "    nvidia-smi --query-compute-apps=pid,used_memory,process_name --format=csv")
except Exception as _e:
    print(f"GPU free: unknown ({type(_e).__name__})")

print(f"TF {tf.__version__} | out: {OUT_DIR}")

GPU free: 3752 MiB
TF 2.21.0 | out: models/vgg_classifier


## Data and split

Loaded in the **same order** as the QRC pipeline — categories `sorted()`, files
`sorted()`, resized to 64×64 — and kept as `uint8` so both variants fit in memory
(float32 conversion happens per split). The split is done on *indices* with the
same two `train_test_split` calls and the same seed, which partitions identically
to splitting the arrays themselves.

If the QRC labels file already exists, the assertion below is the real proof that
these rows are the QRC rows, rather than an argument that they should be.

In [2]:
qrc.download_eurosat()
CATEGORIES = sorted(p.name for p in qrc.DATA_DIR.iterdir() if p.is_dir())

imgs, labels = [], []
for li, cat in enumerate(CATEGORIES):
    for path in sorted((qrc.DATA_DIR / cat).glob("*.jpg")):
        imgs.append(np.array(Image.open(path).convert("RGB").resize((64, 64)), dtype=np.uint8))
        labels.append(li)
X_u8 = np.stack(imgs)                      # (N, 64, 64, 3) uint8
y = np.array(labels)
del imgs
print(f"{len(CATEGORIES)} categories, {X_u8.shape} {X_u8.dtype}")

idx = np.arange(len(y))
i_train, i_temp = train_test_split(idx, test_size=0.2, random_state=SEED, stratify=y)
i_val, i_test = train_test_split(i_temp, test_size=0.5, random_state=SEED, stratify=y[i_temp])
assert (len(i_train), len(i_val), len(i_test)) == (21600, 2700, 2700)

if LABELS_PATH.exists():
    _lab = np.load(LABELS_PATH, allow_pickle=True)
    for name, ii in (("y_train", i_train), ("y_val", i_val), ("y_test", i_test)):
        assert np.array_equal(_lab[name], y[ii]), f"{name} differs from the QRC split"
    assert list(_lab["categories"]) == CATEGORIES
    print(f"split verified identical to {LABELS_PATH}")
else:
    print(f"{LABELS_PATH} not there yet — split matches by construction, unverified")

print(f"train {len(i_train)}  val {len(i_val)}  test {len(i_test)}")

10 categories, (27000, 64, 64, 3) uint8
split verified identical to models/eurosat_qrc/split_labels_seed42.npz
train 21600  val 2700  test 2700


## Variants

One architecture, two input depths. `to_grayscale` is the pipeline's own Rec.601
conversion, so the grayscale CNN sees exactly the pixels the PCA sees — that is
the whole point of this second row.

In [3]:
def make_split(ii, gray):
    '''float32 in [0, 1], shaped (n, 64, 64, 1) for grayscale and (n, 64, 64, 3) for RGB.'''
    x = X_u8[ii].astype(np.float32) / 255.0
    return qrc.to_grayscale(x)[..., None] if gray else x


VARIANTS = {"rgb": dict(gray=False, channels=3), "gray": dict(gray=True, channels=1)}
for tag, v in VARIANTS.items():
    print(f"{tag:>5}: {make_split(i_test[:2], v['gray']).shape}")

  rgb: (2, 64, 64, 3)
 gray: (2, 64, 64, 1)


## Model and training

The three-block VGG-style CNN of the previous notebook, unchanged, so the
reference stays the same model. Two departures, both deliberate:

- **A validation split with early stopping** instead of a flat 5 epochs. A CNN
  stopped early understates the CNN, and understating the CNN is the direction
  that flatters the QRC arms — the opposite of what a reference row is for. The
  patience is 10 for the same reason: at 5 the RGB run was cut at epoch 4 on a
  noisy validation curve, which is not a comparison, it is a coin toss.
- **The model is saved.**

In [4]:
from keras import Input
from keras.callbacks import EarlyStopping
from keras.layers import Conv2D, Dense, Dropout, Flatten, MaxPooling2D
from keras.models import Sequential, load_model


def build_vgg(channels):
    '''Same three-block VGG as Classificaton_EuroSAT_VGG.ipynb, input depth parametrised.'''
    m = Sequential([Input(shape=(64, 64, channels))])
    for filters in (32, 64, 128):
        m.add(Conv2D(filters, (3, 3), activation="relu",
                     kernel_initializer="he_uniform", padding="same"))
        m.add(Conv2D(filters, (3, 3), activation="relu",
                     kernel_initializer="he_uniform", padding="same"))
        m.add(MaxPooling2D((2, 2)))
    m.add(Flatten())
    m.add(Dense(128, activation="relu", kernel_initializer="he_uniform"))
    m.add(Dropout(0.5))
    m.add(Dense(len(CATEGORIES), activation="softmax"))
    return m


def train_or_load(tag, gray, channels):
    model_path = OUT_DIR / f"vgg_{tag}.keras"
    hist_path = OUT_DIR / f"vgg_{tag}_history.json"
    if model_path.exists() and not RETRAIN:
        print(f"{tag}: loading {model_path}")
        return load_model(model_path), json.loads(hist_path.read_text())

    print(f"{tag}: training")
    tf.keras.utils.set_random_seed(SEED)
    model = build_vgg(channels)
    model.compile(optimizer="rmsprop", loss="sparse_categorical_crossentropy",
                  metrics=["accuracy"])
    h = model.fit(make_split(i_train, gray), y[i_train],
                  validation_data=(make_split(i_val, gray), y[i_val]),
                  epochs=EPOCHS, batch_size=BATCH, verbose=2,
                  callbacks=[EarlyStopping(monitor="val_loss", patience=PATIENCE,
                                           restore_best_weights=True)])
    model.save(model_path)
    hist_path.write_text(json.dumps(h.history))
    print(f"{tag}: saved {model_path} ({len(h.history['loss'])} epochs)")
    return model, h.history

In [5]:
models, histories = {}, {}
for tag, v in VARIANTS.items():
    models[tag], histories[tag] = train_or_load(tag, v["gray"], v["channels"])
    best = int(np.argmin(histories[tag]["val_loss"]))
    print(f"{tag:>5}: best epoch {best + 1}, val_loss {histories[tag]['val_loss'][best]:.4f}, "
          f"val_acc {histories[tag]['val_accuracy'][best]:.4f}\n")

rgb: training
Epoch 1/50
675/675 - 19s - 28ms/step - accuracy: 0.4973 - loss: 1.3799 - val_accuracy: 0.7419 - val_loss: 0.6918
Epoch 2/50
675/675 - 9s - 13ms/step - accuracy: 0.7451 - loss: 0.7672 - val_accuracy: 0.8078 - val_loss: 0.5364
Epoch 3/50
675/675 - 9s - 13ms/step - accuracy: 0.8134 - loss: 0.5774 - val_accuracy: 0.8533 - val_loss: 0.4385
Epoch 4/50
675/675 - 9s - 13ms/step - accuracy: 0.8533 - loss: 0.4622 - val_accuracy: 0.8867 - val_loss: 0.3521
Epoch 5/50
675/675 - 9s - 13ms/step - accuracy: 0.8731 - loss: 0.4134 - val_accuracy: 0.8748 - val_loss: 0.3595
Epoch 6/50
675/675 - 9s - 13ms/step - accuracy: 0.8869 - loss: 0.3741 - val_accuracy: 0.8796 - val_loss: 0.3474
Epoch 7/50
675/675 - 9s - 13ms/step - accuracy: 0.8936 - loss: 0.3590 - val_accuracy: 0.8326 - val_loss: 0.6476
Epoch 8/50
675/675 - 9s - 13ms/step - accuracy: 0.8961 - loss: 0.3770 - val_accuracy: 0.8952 - val_loss: 0.4015
Epoch 9/50
675/675 - 9s - 13ms/step - accuracy: 0.8887 - loss: 0.4178 - val_accuracy: 0.8

## Evaluation on the QRC test rows

Macro-F1 is the headline: the classes are imbalanced (2000–3000 images each), so
plain accuracy over-weights the large ones. Per-image predictions are saved
because the paired tests downstream need per-image outcomes, not these summaries.

In [6]:
from sklearn.metrics import (accuracy_score, balanced_accuracy_score,
                             classification_report, f1_score)

y_test = y[i_test]
preds, probs = {}, {}
for tag, v in VARIANTS.items():
    p = models[tag].predict(make_split(i_test, v["gray"]), batch_size=64, verbose=0)
    probs[tag], preds[tag] = p, p.argmax(1)

print(f"{'variant':>8}{'accuracy':>11}{'macro-F1':>11}{'bal. acc':>11}")
print("-" * 41)
for tag in VARIANTS:
    print(f"{tag:>8}{accuracy_score(y_test, preds[tag]):>11.4f}"
          f"{f1_score(y_test, preds[tag], average='macro'):>11.4f}"
          f"{balanced_accuracy_score(y_test, preds[tag]):>11.4f}")

for tag in VARIANTS:
    print(f"\n--- {tag} ---")
    print(classification_report(y_test, preds[tag], target_names=CATEGORIES, digits=3))

 variant   accuracy   macro-F1   bal. acc
-----------------------------------------
     rgb     0.8763     0.8711     0.8705
    gray     0.8559     0.8489     0.8481

--- rgb ---
                      precision    recall  f1-score   support

          AnnualCrop      0.915     0.860     0.887       300
              Forest      0.900     0.993     0.945       300
HerbaceousVegetation      0.762     0.810     0.785       300
             Highway      0.910     0.724     0.806       250
          Industrial      0.889     0.928     0.908       250
             Pasture      0.891     0.855     0.872       200
       PermanentCrop      0.909     0.640     0.751       250
         Residential      0.806     0.997     0.891       300
               River      0.866     0.908     0.887       250
             SeaLake      0.967     0.990     0.979       300

            accuracy                          0.876      2700
           macro avg      0.881     0.871     0.871      2700
        wei

In [7]:
np.savez(OUT_DIR / "vgg_test_predictions.npz",
         test_index=i_test, y_true=y_test,
         categories=np.array(CATEGORIES),
         **{f"pred_{t}": preds[t] for t in VARIANTS},
         **{f"prob_{t}": probs[t] for t in VARIANTS})
print(f"saved {OUT_DIR / 'vgg_test_predictions.npz'}")
for p in sorted(OUT_DIR.iterdir()):
    print(f"  {p.name:<28}{p.stat().st_size / 1e6:>8.1f} MB")

saved models/vgg_classifier/vgg_test_predictions.npz
  vgg_gray.keras                  10.7 MB
  vgg_gray_history.json            0.0 MB
  vgg_rgb.keras                   10.8 MB
  vgg_rgb_history.json             0.0 MB
  vgg_test_predictions.npz         0.3 MB
